# Nemotron 0.88 (nb02 clone)

In [ ]:
import os, sys, glob, subprocess, warnings
warnings.filterwarnings('ignore')
os.environ['PYTHONUNBUFFERED'] = '1'

for pkg_dir in ['/kaggle/input/datasets/mayukh18/nemotron-packages',
                '/kaggle/input/nemotron-packages']:
    p = os.path.join(pkg_dir, 'packages')
    if os.path.isdir(p):
        print(f'Installing from {p}', flush=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--find-links', p,
            'peft', 'transformers', 'datasets', 'accelerate', 'bitsandbytes'], check=True)
        break

for search_dir in [pkg_dir, os.path.join(pkg_dir, 'packages')]:
    for whl in sorted(glob.glob(os.path.join(search_dir, 'mamba_ssm-*.whl'))):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', whl], check=True)
    for whl in sorted(glob.glob(os.path.join(search_dir, 'causal*conv1d*.whl'))):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', whl], check=True)

import mamba_ssm, causal_conv1d, torch
print(f'mamba={mamba_ssm.__version__}, causal={causal_conv1d.__version__}', flush=True)
print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB', flush=True)
print('Setup complete.', flush=True)


## Data Loading + Synthetic Generation

In [ ]:
from pathlib import Path
import gc, json, math, random, re, string
import numpy as np
import pandas as pd

SEED = 3407
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BASE_MODEL_PATH = '/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1'
COMP_DIR = Path('/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge')
train_df = pd.read_csv(COMP_DIR / 'train.csv')

ALPHABET = string.ascii_lowercase
WORD_BANK = ['stone','cable','garden','planet','window','rocket','silver','harbor','market','forest','magnet','bridge']

def prompt_bucket(t):
    t = str(t).lower()
    if re.search(r'\b[01]{4,}\b', t): return 'bit_like'
    if '=' in t or re.search(r'[a-z]\s*[\+\-\*\/]\s*[a-z0-9]', t): return 'equation_like'
    if re.search(r'\b(base|binary|hex|octal|roman)\b', t): return 'numeral_like'
    if re.search(r'\b[a-z]{4,}\b', t) and not re.search(r'\d', t): return 'cipher_like'
    return 'other'

train_df['bucket'] = train_df['prompt'].apply(prompt_bucket)
print(f'Training: {len(train_df)} samples', flush=True)

def format_examples(title, examples, target_input):
    lines = [title, '', 'Infer the hidden rule from the examples and solve the target case.', '']
    for i, (inp, out) in enumerate(examples, 1):
        lines += [f'Example {i}:', f'Input: {inp}', f'Output: {out}', '']
    lines += ['Target:', f'Input: {target_input}', 'Output:']
    return '\n'.join(lines)

rng = random.Random(SEED)
def gen_bit(rng):
    shift = rng.choice([1,2])
    rules = [('inv', lambda s: ''.join('1' if c=='0' else '0' for c in s), 'The rule flips every bit.'),
             ('rev', lambda s: s[::-1], 'The rule reverses the bit string.'),
             ('rot', lambda s,k=shift: s[k:]+s[:k], f'The rule rotates left by {shift}.')]
    _, fn, expl = rng.choice(rules)
    vals = []
    while len(vals)<4:
        s = ''.join(rng.choice('01') for _ in range(rng.choice([6,7,8])))
        if s not in vals: vals.append(s)
    ans = fn(vals[3])
    return {'prompt': format_examples('Bit puzzle', [(s,fn(s)) for s in vals[:3]], vals[3]), 'answer': ans, 'trace': f'{expl} Applying gives \\boxed{{{ans}}}.'}

def gen_cipher(rng):
    sh = rng.choice([1,2,3,4])
    rules = [('rev', lambda w: w[::-1], 'Reverses.'),
             ('cae', lambda w,k=sh: ''.join(ALPHABET[(ALPHABET.index(c)+k)%26] for c in w), f'Caesar +{sh}.'),
             ('atb', lambda w: ''.join(ALPHABET[25-ALPHABET.index(c)] for c in w), 'Atbash.')]
    _, fn, expl = rng.choice(rules)
    ws = rng.sample(WORD_BANK, 4); ans = fn(ws[3])
    return {'prompt': format_examples('Cipher puzzle', [(w,fn(w)) for w in ws[:3]], ws[3]), 'answer': ans, 'trace': f'{expl} Answer \\boxed{{{ans}}}.'}

def gen_unit(rng):
    rules = [('km',1000,'km','m'),('kg',1000,'kg','g'),('hr',60,'hour','min'),('day',24,'day','hr'),('m',100,'m','cm')]
    _,f,s,d = rng.choice(rules); vs = rng.sample(range(2,25),4); ans = str(vs[3]*f)
    return {'prompt': format_examples(f'Unit ({s}->{d})', [(f'{v} {s}',str(v*f)) for v in vs[:3]], f'{vs[3]} {s}'), 'answer': ans, 'trace': f'Factor {f}. \\boxed{{{ans}}}.'}

def gen_numeral(rng):
    rules = [('bin', lambda n: format(n,'b'), 'To binary.'), ('hex', lambda n: format(n,'x'), 'To hex.')]
    _, fn, expl = rng.choice(rules); ns = rng.sample(range(6,40),4); ans = fn(ns[3])
    return {'prompt': format_examples('Numeral puzzle', [(f'{n}',fn(n)) for n in ns[:3]], f'{ns[3]}'), 'answer': ans, 'trace': f'{expl} \\boxed{{{ans}}}.'}

def gen_eq(rng):
    a = rng.choice([1,2,3,4,5,6])
    def mk(): x=rng.choice(range(-8,9)); b=rng.choice(range(-9,10)); return (f'{a}x+{b}={a*x+b}' if b>=0 else f'{a}x-{abs(b)}={a*x+b}', str(x))
    exs = [mk() for _ in range(3)]; tgt = mk()
    return {'prompt': format_examples('Equation puzzle', exs, tgt[0]), 'answer': tgt[1], 'trace': f'x={tgt[1]}. \\boxed{{{tgt[1]}}}.' }

GENS = {'bit_like':gen_bit,'cipher_like':gen_cipher,'unit_like':gen_unit,'numeral_like':gen_numeral,'equation_like':gen_eq}
synth = []
for b,g in GENS.items():
    for _ in range(2400): synth.append(g(rng))
synth_df = pd.DataFrame(synth)
print(f'Synthetic: {len(synth_df)}', flush=True)


## Build Training Records

In [ ]:
def build_user_prompt(prompt):
    return prompt + '\nPlease put your final answer inside \\boxed{}.'

records = []
for _, row in train_df.iterrows():
    p, a = str(row['prompt']), str(row['answer'])
    trace = 'I identify one rule that matches all examples, verify that the same rule is consistent across the prompt, then apply it to the target. Final answer: \\boxed{' + a + '}.'
    records.append({'messages': [
        {'role':'user','content':build_user_prompt(p)},
        {'role':'assistant','content':trace}
    ]})

for _, row in synth_df.iterrows():
    records.append({'messages': [
        {'role':'user','content':build_user_prompt(str(row['prompt']))},
        {'role':'assistant','content':str(row['trace'])}
    ]})

random.shuffle(records)
print(f'Records: {len(records)}', flush=True)


## Model Loading (BNB 4-bit + Warmstart)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True, use_fast=False)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

print('Loading model (4-bit)...', flush=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    trust_remote_code=True,
    quantization_config=quant_config,
    device_map='auto',
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
print(f'Model loaded. VRAM: {torch.cuda.memory_allocated(0)/1e9:.1f}GB', flush=True)

# Load huikang warmstart adapter
WARMSTART_DIR = '/kaggle/input/models/huikang/nemotron-adapter/transformers/default/27'
model = PeftModel.from_pretrained(model, WARMSTART_DIR, is_trainable=True)
print('Warmstart adapter loaded', flush=True)

# Cast LoRA params to bf16 (warmstart saves in fp32, causes dtype mismatch in MoE)
for name, param in model.named_parameters():
    if '.lora_' in name and param.dtype == torch.float32:
        param.data = param.data.to(torch.bfloat16)
print('LoRA params cast to bf16', flush=True)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / Total: {total:,}', flush=True)

# CRITICAL: Disable Mamba fast path (incompatible with BNB 4-bit)
import sys
for _name, _m in sys.modules.items():
    if "modeling_nemotron_h" in _name and hasattr(_m, "is_fast_path_available"):
        _m.is_fast_path_available = False
        print("Mamba fast path DISABLED (BNB 4-bit compat)", flush=True)
        break


## Training (240 steps)

In [ ]:
from transformers import Trainer, TrainingArguments
from torch.utils.data import Dataset as TorchDataset

MAX_LENGTH = 6144
MAX_STEPS = 240

print('Tokenizing...', flush=True)
tokenized_rows = []
for rec in records:
    prompt_msgs = rec['messages'][:1]
    full_msgs = rec['messages']
    prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(full_msgs, tokenize=False, add_generation_prompt=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False, truncation=True, max_length=MAX_LENGTH)['input_ids']
    full_ids = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=MAX_LENGTH)['input_ids']
    labels = ([-100]*len(prompt_ids) + full_ids[len(prompt_ids):])[:len(full_ids)]
    tokenized_rows.append({'input_ids': full_ids, 'attention_mask': [1]*len(full_ids), 'labels': labels})
print(f'Tokenized: {len(tokenized_rows)}', flush=True)

class ListDataset(TorchDataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx): return self.rows[idx]

class DataCollator:
    def __call__(self, features):
        max_len = max(len(x['input_ids']) for x in features)
        input_ids, attention_mask, labels = [], [], []
        for row in features:
            pad = max_len - len(row['input_ids'])
            input_ids.append(row['input_ids'] + [tokenizer.pad_token_id]*pad)
            attention_mask.append(row['attention_mask'] + [0]*pad)
            labels.append(row['labels'] + [-100]*pad)
        return {'input_ids': torch.tensor(input_ids, dtype=torch.long),
                'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
                'labels': torch.tensor(labels, dtype=torch.long)}

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir='/kaggle/working/output',
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=2e-4,
        max_steps=MAX_STEPS,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        bf16=False, fp16=False,
        logging_steps=10,
        save_strategy='no',
        report_to=[],
        remove_unused_columns=False,
        gradient_checkpointing=False,
        optim='paged_adamw_8bit',
    ),
    train_dataset=ListDataset(tokenized_rows),
    data_collator=DataCollator(),
    processing_class=tokenizer,
)

print(f'Training: {MAX_STEPS} steps', flush=True)
trainer.train()
print('Training done', flush=True)


## Save + Package Submission

In [ ]:
import shutil, zipfile
from safetensors.torch import load_file, save_file

ADAPTER_DIR = '/kaggle/working/trained_adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Fix lm_head key naming
st_path = os.path.join(ADAPTER_DIR, 'adapter_model.safetensors')
tensors = load_file(st_path)
renamed = {k.replace('base_model.model.lm_head.', 'base_model.model.backbone.lm_head.'): v for k, v in tensors.items()}
save_file(renamed, st_path)

# Fix config
cfg_path = os.path.join(ADAPTER_DIR, 'adapter_config.json')
with open(cfg_path) as f: cfg = json.load(f)
cfg['base_model_name_or_path'] = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
# Load reference config keys (nb02 0.87 approach)
REFERENCE_ZIP = '/kaggle/input/notebooks/huikang/nvidia-nemotron-all-linear/reference/submission.zip'
if os.path.exists(REFERENCE_ZIP):
    import zipfile as _zf
    with _zf.ZipFile(REFERENCE_ZIP, 'r') as rzf:
        with rzf.open('adapter_config.json') as rf:
            ref_cfg = json.loads(rf.read())
    for key in ['alora_invocation_tokens','arrow_config','ensure_weight_tying','peft_version','qalora_group_size','target_parameters','use_qalora']:
        if key in ref_cfg:
            cfg[key] = ref_cfg[key]
    print(f'Reference config keys copied', flush=True)
cfg['inference_mode'] = True
with open(cfg_path, 'w') as f: json.dump(cfg, f, indent=2)

# Create submission.zip
zip_path = '/kaggle/working/submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in ['adapter_config.json', 'adapter_model.safetensors']:
        zf.write(os.path.join(ADAPTER_DIR, name), arcname=name)

print(f'submission.zip: {os.path.getsize(zip_path)/1024/1024:.1f}MB', flush=True)
print('Done!', flush=True)
